# GeneFlow AI · Entrenamiento completo

Este notebook entrena la mejor configuración de la búsqueda de hiperparámetros con el conjunto de entrenamiento completo. Funciona igual **en local** y **en Kaggle**: detecta dónde se ejecuta y se adapta.

| | Local | Kaggle |
|---|---|---|
| Entorno | El repositorio y el `.venv` actuales, sin instalar nada | Clona el repositorio e instala Python 3.14, las dependencias y PyTorch |
| Datos | Reutiliza `data/` y solo ejecuta `prepare` si falta el dataset | Enlaza el dataset procesado de **Input** (`geneflow-dataset`) y solo genera la caché de tokens; sin él, `geneflow prepare` desde cero (~1 h) |
| Precisión | bf16 | fp16 (la T4 no tiene bf16) |
| Entrenamiento | En primer plano, con la barra de progreso en directo | En segundo plano en las dos GPU, con un informe cada 10 minutos |
| Épocas | `LOCAL_EPOCHS` (10 por defecto) | Las que quepan en la sesión de 12 horas |
| Salidas | `checkpoints/train/` | `/kaggle/working/train/`, en la pestaña **Output** |

## Cómo lanzarlo

- **En local:** abre el notebook con el kernel del proyecto, **reinicia el kernel** y ejecuta todo.
- **En Kaggle:** **Create → New Notebook → File → Import Notebook**; en **Input → Add Input**, añade tu dataset **`geneflow-dataset`**; en **Session options**, **GPU T4 ×2** e **Internet On**; después **Save Version → Save & Run All (Commit)**. Cuando termine, descarga la carpeta **`train`** de **Output**.

## Qué hace

| Paso | Qué ocurre |
|---|---|
| 1 | Entorno |
| 2 | Datos |
| 3 | **Demo**: 2 épocas cortas con la barra de progreso en directo, la accuracy por nivel tras cada validación, el informe de clasificación y las gráficas |
| 4 | Plan: cuántas épocas se entrenan |
| 5 | **Entrenamiento completo** |
| 6 | **Reportes**: gráfica de loss, curvas de accuracy por nivel, comparación entre ejecuciones e informe de clasificación |

## Cómo se ve el entrenamiento

Cada época muestra una barra de progreso al estilo Keras, y al terminar la validación, la pérdida y la accuracy de train y val en cada nivel:

```
Epoch 3/10
10985/10985 [==============================] - 18m 02s 95ms/step - loss: 1.7252 - val_loss: 1.0548 - lr: 6.54e-04
            domain  kingdom   phylum    class    order   family    genus  species
train       0.9990   0.9950   0.9700   0.9400   0.8800   0.8100   0.7000   0.4100
val         0.9990   0.9940   0.9650   0.9300   0.8600   0.7900   0.6680   0.3440
```

Al final se calcula con el mejor checkpoint el **informe de clasificación** sobre `val`: accuracy, precisión, recall y F1 (macro y ponderados), soporte y clases por nivel.

## Qué entrena

| GPU | Configuración | Origen |
|---|---|---|
| 0 | **CNN, prueba 12**: `base`, dos bloques, kernel 5, lr 8,2e-4, dropout 0,05 | `reports/search/cnn-best.json`, la mejor de la búsqueda (0,605) |
| 1 (solo en Kaggle) | CNN, prueba 12 con **otra semilla**, para medir la variabilidad entre ejecuciones | la misma configuración |

Cada época recorre el train completo, **7,97 millones de secuencias** reales y sintéticas, y valida con las **400 mil** de `val`. `test` no se toca. Para aprovechar la segunda GPU con el finalista grande (prueba 20), cambia la segunda entrada de `RUNS` por la comentada.

**En Kaggle**, las épocas se calculan con la velocidad medida en la demo: el 92 % del tiempo que queda entre la duración de una época. Cada entrenamiento recibe además como límite estricto el tiempo de sesión menos 30 minutos; si se acerca, se corta y conserva los checkpoints de las épocas terminadas.

## Qué se guarda

| Archivo | Contenido |
|---|---|
| `<nombre>/best.pt`, `<nombre>/last.pt` | Los pesos de la mejor época (menor pérdida de validación) y de la última |
| `<nombre>/history.json` | Pérdida y accuracy de train y val por nivel en cada época |
| `<nombre>/steps.json` | La pérdida de entrenamiento a lo largo de cada época |
| `<nombre>/report.json` | El informe de clasificación por nivel |
| `<nombre>/run.json`, `<nombre>/label_space.json` | La configuración completa y el espacio de etiquetas, necesarios para evaluar el modelo |
| `<nombre>/train.log` | La salida completa del entrenamiento |
| `reports/*.png`, `reports/*-summary.json` | Las gráficas y los resúmenes de la demo y del entrenamiento completo |


In [ ]:
import json
import math
import os
from pathlib import Path
import shutil
import subprocess
import sys
import time

KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ
SESSION_START = time.time()

RUNS = [
    {"name": "cnn-trial12", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260923, "relative_cost": 1.0},
    {"name": "cnn-trial12-seed2", "params": "reports/search/cnn-best.json", "trial": 12, "seed": 20260924, "relative_cost": 1.0},
    # {"name": "cnn-trial20", "params": "reports/search/cnn-trials.json", "trial": 20, "seed": 20260923, "relative_cost": 1.35},
]

DEMO = {"epochs": 2, "samples_per_epoch": 12_800, "log_every": 20}
LOCAL_EPOCHS = 10
MAX_EPOCHS = 12
EPOCH_BUDGET_FRACTION = 0.92
EVALUATION_SPEEDUP = 3.0
REPORT_EVERY_SECONDS = 600

LEVELS = ["domain", "kingdom", "phylum", "class", "order", "family", "genus", "species"]
OBJECTIVE_LEVELS = ["genus", "species"]

if KAGGLE:
    SESSION_HOURS = 12.0
    SAFETY_MARGIN_HOURS = 0.5
    REPOSITORY = "https://github.com/Dexaroz/geneflow-ai-taxonomy-classifier.git"
    BRANCH = "main"
    WORK = Path("/tmp/geneflow")
    REPO = WORK / "repo"
    DATA = WORK / "data"
    OUTPUT = Path("/kaggle/working/train")
    INPUT = Path("/kaggle/input")
    DEVICE_ARGUMENTS = ["--precision", "fp16", "--gpu-memory-fraction", "0.92"]
else:
    SESSION_HOURS = None
    REPO = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "pyproject.toml").exists())
    WORK = REPO / "checkpoints"
    DATA = REPO / "data"
    OUTPUT = REPO / "checkpoints" / "train"
    INPUT = None
    DEVICE_ARGUMENTS = ["--precision", "bf16", "--gpu-memory-fraction", "0.9"]

BIN = REPO / ".venv" / ("Scripts" if os.name == "nt" else "bin")
GENEFLOW = BIN / ("geneflow.exe" if os.name == "nt" else "geneflow")
PYTHON = BIN / ("python.exe" if os.name == "nt" else "python")
DEMO_DIR = WORK / "demo"
REPORTS = OUTPUT / "reports"

WORK.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)


def run(command, **kwargs):
    print("$", " ".join(str(part) for part in command), flush=True)

    return subprocess.run([str(part) for part in command], check=True, **kwargs)


def child_environment(gpu):
    return dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")


def stream(command, log_path, environment):
    print("$", " ".join(str(part) for part in command), flush=True)

    with open(log_path, "wb") as log:
        process = subprocess.Popen([str(part) for part in command], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=environment)

        while chunk := process.stdout.read1(4096):
            log.write(chunk)
            sys.stdout.write(chunk.decode("utf-8", errors="replace"))
            sys.stdout.flush()

    if process.wait() != 0:
        raise RuntimeError(f"El entrenamiento terminó con código {process.returncode}; revisa {log_path}")


def elapsed_hours():
    return (time.time() - SESSION_START) / 3600


def remaining_hours():
    if SESSION_HOURS is None:
        return math.inf

    return SESSION_HOURS - SAFETY_MARGIN_HOURS - elapsed_hours()


print(f"{'Kaggle' if KAGGLE else 'Local'} | repositorio {REPO} | datos {DATA} | salidas {OUTPUT}")

## 1. Entorno

In [ ]:
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=True,
)
gpus = [line.split(", ") for line in gpu_query.stdout.strip().splitlines()]

for index, (name, driver, memory) in enumerate(gpus):
    print(f"GPU {index}: {name} | driver {driver} | {memory}")

if KAGGLE:
    cuda_index = "cu130" if int(gpus[0][1].split(".")[0]) >= 580 else "cu126"

    run([sys.executable, "-m", "pip", "install", "--quiet", "uv"])

    if REPO.exists():
        shutil.rmtree(REPO)

    run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY, REPO])
    run(["uv", "python", "install", "3.14"])
    run(["uv", "sync", "--frozen", "--no-default-groups", "--no-install-package", "torch"], cwd=REPO)
    run(["uv", "pip", "install", "--python", PYTHON, "torch==2.14.0", "--index-url", f"https://download.pytorch.org/whl/{cuda_index}"], cwd=REPO)

run(["git", "-C", REPO, "log", "-1", "--format=%h %s"])
run([PYTHON, "-c", "import torch; print('torch', torch.__version__, '| GPU visibles:', torch.cuda.device_count(), '|', torch.cuda.get_device_name(0))"])

print(f"Entorno listo en {elapsed_hours() * 60:.0f} min")

## 2. Datos

In [ ]:
PROCESSED = DATA / "processed" / "geneflow"
manifests = INPUT.rglob("manifest.json") if INPUT is not None and INPUT.exists() else []
mounted = next((path for path in manifests if json.loads(path.read_text()).get("dataset") == "geneflow"), None)

if mounted is not None:
    manifest = json.loads(mounted.read_text())
    PROCESSED.mkdir(parents=True, exist_ok=True)

    for name in manifest["files"]:
        link = PROCESSED / name

        if not link.exists():
            link.symlink_to(mounted.parent / name)

    print(f"Dataset de Input: {mounted.parent} | commit {manifest['commit'][:7]} | {manifest['sequences']:,} secuencias")

elif not (PROCESSED / "train_synthetic.parquet").exists():
    run([GENEFLOW, "prepare", "--data-dir", DATA])

run([GENEFLOW, "cache", "--data-dir", DATA])

train_sequences = json.loads((DATA / "interim" / "geneflow" / "train_tokens.json").read_text())["sequences"]
validation_sequences = json.loads((DATA / "interim" / "geneflow" / "val_tokens.json").read_text())["sequences"]

print(f"Train: {train_sequences:,} secuencias | val: {validation_sequences:,}")
print(f"Datos listos a los {elapsed_hours() * 60:.0f} min")

## Funciones de reporte

El mismo reporte sirve para la demo y para el entrenamiento completo:

- **Gráfica de loss**: la pérdida de entrenamiento a lo largo de cada época (media móvil de cada tramo de lotes) y la de validación al final de cada época.
- **Objetivo de la búsqueda** (media de la accuracy de género y especie) por época, frente a la mejor prueba de la búsqueda.
- **Accuracy por nivel** en cada época, de train (discontinua) y val (continua).
- **Accuracy por nivel en la mejor época** de cada ejecución, lado a lado.
- **Informe de clasificación** por nivel con el mejor checkpoint.

Las figuras y un resumen en JSON se guardan en `reports/`.

In [ ]:
import matplotlib.pyplot as plt

SEARCH_BEST = json.loads((REPO / "reports" / "search" / "cnn-best.json").read_text())["value"]
REPORT_COLUMNS = {
    "accuracy": "accuracy",
    "macro_precision": "macro-p",
    "macro_recall": "macro-r",
    "macro_f1": "macro-f1",
    "weighted_precision": "weighted-p",
    "weighted_recall": "weighted-r",
    "weighted_f1": "weighted-f1",
}


def objective(accuracy):
    return sum(accuracy[level] for level in OBJECTIVE_LEVELS) / len(OBJECTIVE_LEVELS)


def read_json(path):
    return json.loads(path.read_text()) if path.exists() else None


def windowed_losses(steps):
    points = []
    previous = {}

    for record in steps:
        epoch, step, loss = record["epoch"], record["step"], record["loss"]
        last_step, last_total = previous.get(epoch, (0, 0.0))
        points.append((epoch, step, (loss * step - last_total) / (step - last_step)))
        previous[epoch] = (step, loss * step)

    last_steps = {epoch: step for epoch, step, _ in points}

    return [(epoch - 1 + step / last_steps[epoch], loss) for epoch, step, loss in points]


def loss_figure(runs, prefix):
    figure, axis = plt.subplots(figsize=(14, 5))
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

    for (name, directory), color in zip(runs.items(), colors):
        history = read_json(directory / "history.json") or []
        steps = read_json(directory / "steps.json") or []
        points = windowed_losses(steps)

        if points:
            axis.plot([x for x, _ in points], [y for _, y in points], color=color, alpha=0.8, linewidth=1.2, label=f"{name} · train (por lotes)")

        axis.plot([record["epoch"] for record in history], [record["train_loss"] for record in history], "s", color=color, markersize=6, label=f"{name} · train (época)")
        axis.plot([record["epoch"] for record in history], [record["val_loss"] for record in history], "o--", color=color, markersize=8, markerfacecolor="white", label=f"{name} · val")

    axis.set(title=f"Loss · {prefix}", xlabel="época", ylabel="pérdida jerárquica")
    axis.grid(alpha=0.3)
    axis.legend(fontsize=8)
    figure.tight_layout()
    figure.savefig(REPORTS / f"{prefix}-loss.png", dpi=150)
    plt.show()


def accuracy_figure(histories, prefix):
    figure, axes = plt.subplots(1, 3, figsize=(18, 5))
    objective_axis, level_axis, best_axis = axes
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    level_colors = plt.cm.viridis([index / (len(LEVELS) - 1) for index in range(len(LEVELS))])

    for (name, history), color in zip(histories.items(), colors):
        objective_axis.plot([record["epoch"] for record in history], [objective(record["val_accuracy"]) for record in history], "o-", color=color, label=name)

    objective_axis.axhline(SEARCH_BEST, color="grey", linestyle=":", label=f"mejor de la búsqueda ({SEARCH_BEST:.3f})")

    first_name, first_history = next(iter(histories.items()))
    epochs = [record["epoch"] for record in first_history]

    for level, color in zip(LEVELS, level_colors):
        level_axis.plot(epochs, [record["val_accuracy"][level] for record in first_history], "o-", color=color, label=level)

        if first_history[0].get("train_accuracy"):
            level_axis.plot(epochs, [record["train_accuracy"][level] for record in first_history], "--", color=color, alpha=0.6)

    bests = {name: min(history, key=lambda record: record["val_loss"]) for name, history in histories.items()}
    width = 0.8 / len(bests)

    for offset, ((name, best), color) in enumerate(zip(bests.items(), colors)):
        positions = [index + offset * width - 0.4 + width / 2 for index in range(len(LEVELS))]
        best_axis.bar(positions, [best["val_accuracy"][level] for level in LEVELS], width, color=color, label=f"{name} (época {best['epoch']})")

    objective_axis.set(title="Objetivo: media de género y especie (val)", xlabel="época", ylabel="accuracy", ylim=(0, 1))
    level_axis.set(title=f"Accuracy por nivel · {first_name} (val continua, train discontinua)", xlabel="época", ylim=(0, 1))
    best_axis.set(title="Accuracy de val por nivel en la mejor época", ylim=(0, 1))
    best_axis.set_xticks(range(len(LEVELS)), LEVELS, rotation=30)

    for axis in axes:
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)

    for axis in (objective_axis, level_axis):
        axis.xaxis.get_major_locator().set_params(integer=True)

    figure.tight_layout()
    figure.savefig(REPORTS / f"{prefix}-accuracy.png", dpi=150)
    plt.show()

    return bests


def print_classification_report(name, directory):
    rows = read_json(directory / "report.json")

    if not rows:
        return

    print(f"\nInforme de clasificación · {name} (mejor checkpoint, val)\n")
    print(f"{'':>10}" + "".join(f"{title:>12}" for title in REPORT_COLUMNS.values()) + f"{'support':>10}{'classes':>9}")

    for row in rows:
        print(f"{row['level']:>10}" + "".join(f"{row[column]:>12.4f}" for column in REPORT_COLUMNS) + f"{row['support']:>10}{row['classes']:>9}")

    markers = read_json(directory / "report_by_marker.json")

    if not markers:
        return

    grouped = {"all": rows, **markers}

    for metric, title in (("accuracy", "Accuracy"), ("macro_f1", "Macro-F1")):
        print(f"\n{title} por marcador · {name}\n")
        print(f"{'':>10}" + "".join(f"{level:>10}" for level in LEVELS) + f"{'support':>10}")

        for marker, levels in grouped.items():
            cells = "".join("         -" if math.isnan(level[metric]) else f"{level[metric]:>10.4f}" for level in levels)
            print(f"{marker:>10}{cells}{levels[0]['support']:>10}")


def training_report(runs, prefix, *, classification=True):
    histories = {name: read_json(directory / "history.json") for name, directory in runs.items()}
    histories = {name: history for name, history in histories.items() if history}

    if not histories:
        print("Sin épocas terminadas: no hay nada que reportar")

        return None

    loss_figure(runs, prefix)
    bests = accuracy_figure(histories, prefix)

    summary = {
        name: {
            "best_epoch": best["epoch"],
            "epochs": len(histories[name]),
            "val_loss": best["val_loss"],
            "objective": objective(best["val_accuracy"]),
            "val_accuracy": best["val_accuracy"],
            "minutes": sum(record["seconds"] for record in histories[name]) / 60,
        }
        for name, best in bests.items()
    }
    (REPORTS / f"{prefix}-summary.json").write_text(json.dumps(summary, indent=2))

    header = f"{'ejecución':<22} {'época':>5} {'val loss':>9} {'objetivo':>9} " + " ".join(f"{level[:7]:>7}" for level in LEVELS) + f" {'min':>6}"
    print(header)
    print("-" * len(header))

    for name, row in summary.items():
        print(
            f"{name:<22} {row['best_epoch']:>2}/{row['epochs']:<2} {row['val_loss']:>9.4f} {row['objective']:>9.4f} "
            + " ".join(f"{row['val_accuracy'][level]:>7.3f}" for level in LEVELS)
            + f" {row['minutes']:>6.0f}"
        )

    print(f"\nReferencia: mejor prueba de la búsqueda {SEARCH_BEST:.4f} (6 épocas × 100 mil secuencias, val de 20 mil)")

    if classification:
        for name, directory in runs.items():
            print_classification_report(name, directory)

    return summary


def train_command(entry, directory, epochs, *extra):
    return [
        GENEFLOW, "train",
        "--data-dir", DATA,
        "--params", REPO / entry["params"],
        "--trial", str(entry["trial"]),
        "--output", directory,
        "--epochs", str(epochs),
        "--seed", str(entry["seed"]),
        *DEVICE_ARGUMENTS,
        *extra,
    ]

## 3. Demo: un entrenamiento corto en directo

La misma configuración que el entrenamiento completo, durante 2 épocas de 12.800 secuencias. Con tan pocas secuencias la accuracy de los niveles profundos será baja: lo que importa es ver la barra de progreso, que la pérdida baja y que los reportes se generan.

In [ ]:
demo_run = RUNS[0]

if DEMO_DIR.exists():
    shutil.rmtree(DEMO_DIR)

DEMO_DIR.mkdir(parents=True)

stream(
    train_command(
        demo_run,
        DEMO_DIR,
        DEMO["epochs"],
        "--samples-per-epoch", str(DEMO["samples_per_epoch"]),
        "--log-every", str(DEMO["log_every"]),
        "--refresh-seconds", "0.5",
    ),
    DEMO_DIR / "train.log",
    child_environment(0),
)

demo_parameters = json.loads((DEMO_DIR / "run.json").read_text())["parameters"]
demo_speed = json.loads((DEMO_DIR / "steps.json").read_text())[-1]["sequences_per_second"]

print(f"\nModelo: {demo_parameters['total']:,} parámetros ({demo_parameters['encoder']:,} en el encoder)")
print(f"Velocidad medida: {demo_speed:.0f} secuencias/s\n")

demo_summary = training_report({"demo": DEMO_DIR}, "demo", classification=False)

## 4. Plan de épocas

In [ ]:
active_runs = RUNS[: len(gpus)] if KAGGLE else RUNS[:1]


def epoch_minutes(entry):
    speed = demo_speed / entry["relative_cost"]
    training = train_sequences / speed
    evaluation = validation_sequences / (speed * EVALUATION_SPEEDUP)

    return (training + evaluation) / 60


for entry in active_runs:
    minutes = epoch_minutes(entry)

    if KAGGLE:
        entry["epochs"] = max(1, min(MAX_EPOCHS, math.floor(remaining_hours() * 60 * EPOCH_BUDGET_FRACTION / minutes)))
    else:
        entry["epochs"] = LOCAL_EPOCHS

    print(
        f"{entry['name']}: {entry['epochs']} épocas de ~{minutes:.0f} min "
        f"(~{entry['epochs'] * minutes / 60:.1f} h, {entry['epochs'] * train_sequences / 1e6:.1f} M secuencias)"
    )

if KAGGLE:
    print(f"Tiempo disponible: {remaining_hours():.2f} h")

## 5. Entrenamiento completo

En local se entrena en primer plano con la barra de progreso en directo. En Kaggle se lanza una ejecución por GPU en segundo plano y cada 10 minutos se informa de la última época terminada; la salida completa de cada una queda en su `train.log`.

In [ ]:
def budget_arguments():
    return ["--hours", f"{remaining_hours():.3f}"] if math.isfinite(remaining_hours()) else []


def launch(entry, gpu):
    directory = OUTPUT / entry["name"]
    directory.mkdir(parents=True, exist_ok=True)
    log = open(directory / "train.log", "ab")
    command = train_command(entry, directory, entry["epochs"], "--refresh-seconds", "30", *budget_arguments())

    print(f"Lanzando {entry['name']} en la GPU {gpu}: {entry['epochs']} épocas")

    return subprocess.Popen([str(part) for part in command], stdout=log, stderr=subprocess.STDOUT, env=child_environment(gpu))


def progress(entry):
    history = read_json(OUTPUT / entry["name"] / "history.json")

    if not history:
        return "sin épocas terminadas"

    last = history[-1]
    accuracy = last["val_accuracy"]

    return (
        f"época {last['epoch']}/{entry['epochs']} | val loss {last['val_loss']:.4f} | "
        + " ".join(f"{level[:4]} {accuracy[level]:.3f}" for level in LEVELS)
    )


if KAGGLE:
    processes = {entry["name"]: launch(entry, gpu) for gpu, entry in enumerate(active_runs)}

    while any(process.poll() is None for process in processes.values()):
        time.sleep(REPORT_EVERY_SECONDS)

        print(f"\n[{elapsed_hours():.2f} h de sesión, quedan {max(0.0, remaining_hours()):.2f} h]")

        for entry in active_runs:
            print(f"  {entry['name']}: {progress(entry)}")

    for name, process in processes.items():
        print(f"{name} terminó con código {process.returncode}")

else:
    for entry in active_runs:
        directory = OUTPUT / entry["name"]
        directory.mkdir(parents=True, exist_ok=True)

        stream(
            train_command(entry, directory, entry["epochs"], "--refresh-seconds", "0.5"),
            directory / "train.log",
            child_environment(0),
        )

print(f"\nEntrenamiento terminado a las {elapsed_hours():.2f} h")

## 6. Reportes del entrenamiento completo

La gráfica de loss, las curvas de accuracy, la tabla de la mejor época y el informe de clasificación de cada ejecución. Con dos semillas, la diferencia entre ambas da una idea del ruido entre ejecuciones de la misma configuración.

In [ ]:
final_summary = training_report({entry["name"]: OUTPUT / entry["name"] for entry in active_runs}, "final")

if final_summary and len(final_summary) > 1:
    objectives = [row["objective"] for row in final_summary.values()]
    print(f"\nDiferencia de objetivo entre ejecuciones: {max(objectives) - min(objectives):.4f}")

print("\nArchivos generados:")

for path in sorted(OUTPUT.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(OUTPUT.parent)} ({path.stat().st_size / 1e6:.1f} MB)")